# Square-QDM numerical evidence for the current ICQMBS draft

This notebook is keyed to the numerical notes in **Secs. V, VI, and VIII** of the July 20, 2026 draft.  It uses the square-lattice QDM to supply the missing finite-size scorecard, the compact-versus-collective cancellation diagnostics, deformation data, and a conservative **fixed-width (quasi-one-dimensional)** thermodynamic analysis.

The notebook deliberately separates four statements:

1. **finite-size exactness:** the Type-I shell boundary matrix has the claimed nullity and the reconstructed states have negligible internal and leakage residuals;
2. **local reducibility:** eight compact records close inside bounded plaquette windows, whereas the ninth record requires the full $4\times4$ plaquette set in the present local-operator basis;
3. **deformation stability:** the compact span has a larger preserving coefficient space than the collective mode, but the preserving codimension grows linearly along the strip sequence;
4. **quasi-1D ETH evidence:** an exactly repeated $4N\times4$ cage sequence has a fixed local witness with positive beta-zero activity, while several obstructions prevent interpreting this as a genuine two-dimensional theorem.

The current patch does **not** search for a new $L_x,L_y\to\infty$ construction.  A disabled final cell exposes the more expensive collective-extension search for later use.


## Imports, reproducibility, and figure style

`USE_TEX=False` keeps the notebook runnable without a TeX installation.  Set it to `True` to render labels with the same LaTeX math font as the REVTeX manuscript.  The plotting cells are intentionally simple so their visual style can be adjusted manually.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as scipy_linalg
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    LocalQDMCageSearchConfig,
    Quasi1DSequencePoint,
    RobustQDMLocalCageSearchConfig,
    SquareQDMPeriodicProductUnitCell,
    adjacent_gap_ratio_report,
    audit_quasi_1d_sequence,
    beta_zero_matching_subspace,
    cage_finite_size_scorecard,
    certify_local_witness_on_square_qdm_periodic_sequence,
    certify_square_qdm_periodic_product_sequence,
    classify_cage_state,
    diagnose_boundary_cancellation_matroid,
    diagnose_eigenpair,
    eigenstate_expectations,
    evaluate_square_qdm_classification_witnesses_on_strips,
    local_witnesses_from_classification_report,
    operator_coefficient_compatibility,
    partition_cage_hamiltonian,
    project_coefficients_to_beta_zero_match,
    regional_cage_quotient,
    robust_qdm_local_cage_search,
    scan_square_qdm_beta_zero_energy_density,
    scan_square_qdm_collective_locality_extension,
    scan_square_qdm_periodic_product_cancellation_scaling,
    scan_windowed_operator_annihilators,
    select_microcanonical_window_by_count,
    subspace_complement_basis,
)
from qlinks.models import SquareQDMModel

TOL = 1.0e-10
RANK_TOL = 1.0e-9
RANDOM_SEED = 73291
USE_TEX = False
SAVE_FIGURES = True
SAVE_PDF = False

DATA_DIR = REPO_ROOT / "experimental" / "data" / "square_qdm_draft_evidence"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "text.usetex": USE_TEX,
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.7,
    "lines.linewidth": 1.1,
    "lines.markersize": 4.0,
    "savefig.bbox": "tight",
})
if USE_TEX:
    mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath,amssymb,bm}"


def save_figure(fig, stem):
    if SAVE_FIGURES:
        if SAVE_PDF:
            fig.savefig(FIGURE_DIR / f"{stem}.pdf", pad_inches=0.02)
        fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=300, pad_inches=0.02)
    plt.close(fig)


def embedded_state(record, hilbert_size):
    state = np.zeros(hilbert_size, dtype=np.complex128)
    state[np.asarray(record.cage_state.support, dtype=np.int64)] = record.cage_state.local_state
    return state

print({"repository": str(REPO_ROOT), "data_directory": str(DATA_DIR)})

## 1. Reconstruct the $4\times4$ Type-I census

The draft asks for the exact sector, shell dimensions, boundary shapes, rank/nullity, residual tolerances, support sizes, and degeneracy handling.  The IPR basis selection is used only after the invariant nullspace has been certified.

In [ ]:
square_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)

t0 = time.perf_counter()
square_build = square_model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
)
build_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
square_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=TOL,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=64,
        ipr_candidate_count=32,
        ipr_random_seed=1234,
    ),
).run()
search_seconds = time.perf_counter() - t0

records_04 = tuple(square_search[(0, 4)])
record_06 = square_search[(0, 6), 0]
states_04 = np.column_stack(
    [embedded_state(record, square_search.hilbert_size) for record in records_04]
)

{
    "winding_sector": (0, 0),
    "hilbert_dimension": square_search.hilbert_size,
    "counts_by_signature": square_search.counts_by_signature,
    "build_seconds": build_seconds,
    "search_seconds": search_seconds,
}

In [ ]:
scorecard_rows = []
for signature, records in (((0, 4), records_04), ((0, 6), (record_06,))):
    for record_index, record in enumerate(records):
        full_state = embedded_state(record, square_search.hilbert_size)
        scorecard = cage_finite_size_scorecard(
            square_build.hamiltonian,
            record.candidate.vertices,
            full_state,
            kinetic=square_build.kinetic,
            actual_support=record.cage_state.support,
            amplitude_tolerance=TOL,
            rank_tolerance=TOL,
            metadata={
                "signature": str(signature),
                "record": record_index,
                "basis_strategy": "IPR postselection",
                "winding_x": 0,
                "winding_y": 0,
            },
        )
        scorecard_rows.append(scorecard.to_summary_dict())

scorecard_table = pd.DataFrame(scorecard_rows)
scorecard_table.to_csv(DATA_DIR / "qdm_4x4_type1_scorecard.csv", index=False)
display(scorecard_table[[
    "signature", "record", "candidate_shell_size", "boundary_rows",
    "boundary_columns", "boundary_rank", "boundary_nullity",
    "boundary_singular_gap", "actual_support_size",
    "internal_residual", "boundary_residual", "relative_eigenpair_residual",
]])

The $(0,4)$ shell is a $84\times48$ boundary problem with nullity nine.  Eight IPR representatives have support four, while the ninth has support 48.  The $(0,6)$ shell is a $100\times32$ boundary problem with nullity one.

## 2. Basis-independent $9=8+1$ decomposition

The compact subspace is defined as the span of the eight support-four records.  The quotient of the complete $(0,4)$ cage manifold by this compact span is one dimensional, and its vector has unit overlap with IPR record 8.

In [ ]:
regional_supports = tuple(record.cage_state.support for record in records_04[:8])
quotient_report = regional_cage_quotient(
    square_build.kinetic,
    regional_supports,
    states_04,
    tolerance=TOL,
)
quotient_overlaps = np.abs(states_04.conj().T @ quotient_report.quotient_basis) ** 2

full_support = tuple(
    sorted(set().union(*(set(record.cage_state.support) for record in records_04)))
)
full_blocks = partition_cage_hamiltonian(square_build.kinetic, full_support)
full_column = {basis_index: column for column, basis_index in enumerate(full_support)}
regional_columns = tuple(
    tuple(full_column[basis_index] for basis_index in support)
    for support in regional_supports
)
matroid_report = diagnose_boundary_cancellation_matroid(
    full_blocks.boundary,
    regional_columns,
    tolerance=TOL,
)

quotient_table = pd.DataFrame([{
    **quotient_report.to_summary_dict(),
    "collective_record_overlap": float(quotient_overlaps[8, 0]),
    "regional_circuit_count": matroid_report.regional_circuit_count,
    "weighted_relative_dependency_dimension": matroid_report.relative_dependency_dimension,
}])
quotient_table.to_csv(DATA_DIR / "qdm_4x4_compact_collective_quotient.csv", index=False)
display(quotient_table.T)

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.35))
ax.bar(["complete cage\nmanifold", "compact span", "collective\nquotient"], [9, 8, 1])
ax.set_ylabel("Dimension")
ax.set_ylim(0, 10)
ax.set_title(r"$(0,4)$ cage-space decomposition")
save_figure(fig, "qdm_4x4_9_equals_8_plus_1")
plt.show()

## 3. Minimal local cancellation radius

For each plaquette kinetic term $K_p$, form the action vector $K_p|\psi\rangle$.  Inside each periodic real-space window, minimize the action norm over coefficient vectors of unit Euclidean norm.  The resulting smallest singular value is a coefficient-normalized annihilation residual.

This is a direct numerical implementation of the draft note requesting the residual versus allowed real-space radius.  It does not assume that the optimal coefficients are pairwise or equal-weight.

In [ ]:
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
kinetic_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.kinetic_operators
)
potential_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.potential_operators
)
all_local_term_matrices = kinetic_term_matrices + potential_term_matrices
plaquette_centers = tuple(
    tuple(float(value) for value in square_model.lattice.plaquette_anchor_cell(int(pid)))
    for pid in square_model.plaquette_ids()
)

radius_scans = {}
for label, state in (
    ("compact record 0", states_04[:, 0]),
    ("collective record 8", states_04[:, 8]),
):
    radius_scans[label] = scan_windowed_operator_annihilators(
        kinetic_term_matrices,
        state,
        plaquette_centers,
        radii=(0, 1, 2),
        periodic_box=(4, 4),
        metric="chebyshev",
        normalize_actions=True,
        action_tolerance=1.0e-12,
        rank_tolerance=TOL,
    )

radius_rows = [
    {"state": label, **point.to_summary_dict()}
    for label, report in radius_scans.items()
    for point in report.points
]
radius_table = pd.DataFrame(radius_rows)
radius_table.to_csv(DATA_DIR / "qdm_4x4_minimum_annihilator_radius.csv", index=False)
display(radius_table[[
    "state", "radius", "minimum_residual", "n_active", "rank", "nullity",
    "coefficient_support_size", "active_operator_indices",
]])

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
for label, report in radius_scans.items():
    ax.semilogy(
        [point.radius for point in report.points],
        [max(point.minimum_residual, 1.0e-16) for point in report.points],
        marker="o",
        label=label,
    )
ax.axhline(TOL, linestyle="--", linewidth=0.8, label="numerical tolerance")
ax.set_xlabel("Allowed Chebyshev radius")
ax.set_ylabel("Minimum annihilation residual")
ax.set_xticks([0, 1, 2])
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_annihilator_radius")
plt.show()

The compact representative first reaches the numerical kernel at radius one using two active plaquette terms.  The collective quotient retains a residual about $0.675$ at radius one and reaches a kernel only when all 16 plaquette terms are available.  On the $4\times4$ torus, this is evidence for a system-scale cancellation, not a proof that its radius must diverge on every possible continuation.

## 4. Deformation compatibility and boundary singular gap

The perturbation basis contains 16 plaquette kinetic terms and 32 orientation-resolved flippability projectors.  The direct linear map stacks $(1-|\psi_j\rangle\langle\psi_j|)O_a|\psi_j\rangle$ for the eight compact records.  For the collective state it uses one target vector.  This efficient construction reproduces the draft's 24-versus-11 preserving dimensions.

In [ ]:
compact_compatibility = operator_coefficient_compatibility(
    all_local_term_matrices,
    states_04[:, :8],
    mode="fixed_vectors",
    tolerance=TOL,
)
collective_compatibility = operator_coefficient_compatibility(
    all_local_term_matrices,
    states_04[:, 8],
    mode="fixed_vectors",
    tolerance=TOL,
)
manifold_compatibility = operator_coefficient_compatibility(
    all_local_term_matrices,
    states_04,
    mode="invariant_subspace",
    tolerance=TOL,
)

compatibility_table = pd.DataFrame([
    {"target": "eight compact vectors", **compact_compatibility.to_summary_dict()},
    {"target": "collective quotient vector", **collective_compatibility.to_summary_dict()},
    {"target": "complete nine-dimensional span", **manifold_compatibility.to_summary_dict()},
])
compatibility_table.to_csv(DATA_DIR / "qdm_4x4_deformation_compatibility.csv", index=False)
display(compatibility_table)

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.35))
ax.bar(
    compatibility_table["target"],
    compatibility_table["compatible_dimension"],
)
ax.set_ylabel("Preserving coefficient dimension")
ax.set_ylim(0, 48)
ax.tick_params(axis="x", rotation=18)
save_figure(fig, "qdm_4x4_preserving_dimensions")
plt.show()

In [ ]:
def combine_local_terms(coefficients):
    result = all_local_term_matrices[0] * 0.0
    for coefficient, matrix in zip(coefficients, all_local_term_matrices, strict=True):
        result = result + coefficient * matrix
    return result

compact_only_basis = subspace_complement_basis(
    compact_compatibility.compatible_basis,
    collective_compatibility.compatible_basis,
    tolerance=1.0e-8,
)

# Select a compact-only direction with the largest response on the collective mode.
compact_only_candidates = tuple(
    combine_local_terms(compact_only_basis[:, index])
    for index in range(compact_only_basis.shape[1])
)
collective_responses = np.asarray([
    diagnose_eigenpair(candidate, states_04[:, 8]).residual_norm
    for candidate in compact_only_candidates
])
compact_only_perturbation = compact_only_candidates[int(np.argmax(collective_responses))]

# Select an invariant-manifold direction with a visible boundary deformation.
manifold_candidates = tuple(
    combine_local_terms(manifold_compatibility.compatible_basis[:, index])
    for index in range(manifold_compatibility.compatible_basis.shape[1])
)
manifold_boundary_norms = np.asarray([
    np.linalg.norm(partition_cage_hamiltonian(candidate, full_support).boundary.toarray())
    for candidate in manifold_candidates
])
manifold_perturbation = manifold_candidates[int(np.argmax(manifold_boundary_norms))]

projector_04 = states_04 @ states_04.conj().T
lambdas = (0.0, 1.0e-3, 1.0e-2, 0.1, 0.25, 0.5, 1.0)
deformation_rows = []
for path_label, perturbation in (
    ("complete-manifold preserving", manifold_perturbation),
    ("compact-only preserving", compact_only_perturbation),
):
    for parameter in lambdas:
        deformed = square_build.hamiltonian + parameter * perturbation
        boundary = partition_cage_hamiltonian(deformed, full_support).boundary.toarray()
        singular_values = scipy_linalg.svdvals(boundary)
        rank = int(np.sum(singular_values > RANK_TOL))
        positive = singular_values[singular_values > RANK_TOL]
        gap = float(np.min(positive)) if positive.size else np.inf
        compact_residual = max(
            diagnose_eigenpair(deformed, states_04[:, index]).residual_norm
            for index in range(8)
        )
        collective_residual = diagnose_eigenpair(deformed, states_04[:, 8]).residual_norm
        subspace_leakage = float(np.linalg.norm(
            (np.eye(square_search.hilbert_size) - projector_04)
            @ (deformed @ states_04)
        ))
        deformation_rows.append({
            "path": path_label,
            "parameter": parameter,
            "boundary_nullity": int(boundary.shape[1] - rank),
            "boundary_singular_gap": gap,
            "maximum_compact_residual": compact_residual,
            "collective_residual": collective_residual,
            "nine_dimensional_subspace_leakage": subspace_leakage,
        })

deformation_table = pd.DataFrame(deformation_rows)
deformation_table.to_csv(DATA_DIR / "qdm_4x4_deformation_paths.csv", index=False)
display(deformation_table)

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
for path, group in deformation_table.groupby("path"):
    ax.plot(group["parameter"], group["boundary_singular_gap"], marker="o", label=path)
ax.set_xlabel(r"Deformation strength $\lambda$")
ax.set_ylabel(r"Boundary singular gap $\Delta_B$")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_boundary_gap_paths")
plt.show()

fig, ax = plt.subplots(figsize=(3.35, 2.45))
group = deformation_table[deformation_table["path"] == "compact-only preserving"]
ax.semilogy(group["parameter"], np.maximum(group["maximum_compact_residual"], 1e-16), marker="o", label="compact records")
ax.semilogy(group["parameter"], np.maximum(group["collective_residual"], 1e-16), marker="o", label="collective record")
ax.set_xlabel(r"Deformation strength $\lambda$")
ax.set_ylabel("Fixed-vector residual")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_compact_only_deformation")
plt.show()

The complete-manifold path keeps boundary nullity nine and a finite boundary gap.  A compact-only direction leaves all eight compact vectors exact but immediately lowers the boundary nullity to eight and opens the collective residual linearly.  The result is a deformation-space distinction, not yet a discrete topological invariant.

## 5. Exact $4N\times4$ product sequence

The robust local search reconstructs two independently caged stripe blocks in the $4\times4$ unit cell.  We choose a certificate that repeats along $x$, so the family has geometry $(4N)\times4$.  The local-action certificate proves all repeat counts, while explicit boundary matrices are materialized here only for $N=1,2,3$.

In [ ]:
local_search_config = LocalQDMCageSearchConfig(
    halo_layers=0,
    boundary_mode="relaxed",
    prune_inactive_local_basis_states=True,
    tolerance=TOL,
    degenerate_basis_strategy="ipr",
    ipr_random_seed=1234,
)
robust_config = RobustQDMLocalCageSearchConfig(
    local_config=local_search_config,
    region_strategies=("stripe",),
    stripe_widths=(1,),
    stripe_directions=(0, 1),
    max_regions_per_strategy=None,
    block_signatures=((0, 2),),
    max_records_per_region=2,
    min_blocks=2,
    max_blocks=None,
    max_product_support_size=2048,
    max_paddings_per_stage=100,
    max_paddings_per_packing=10,
    include_sectors=True,
    padding_stages=("static",),
    tolerance=1.0e-9,
    store_full_states=False,
)

stripe_certified, stripe_context = robust_qdm_local_cage_search(
    square_model,
    config=robust_config,
    return_context=True,
)

repeatable_candidates = []
for report_index, report in enumerate(stripe_certified.reports):
    try:
        candidate = SquareQDMPeriodicProductUnitCell.from_padding(
            square_model,
            stripe_context.blocks,
            report.padding,
            repeat_axis="x",
        )
        certificate = certify_square_qdm_periodic_product_sequence(candidate)
    except ValueError:
        continue
    if certificate.is_certified:
        repeatable_candidates.append((report_index, candidate, certificate))

if not repeatable_candidates:
    raise RuntimeError("No x-repeatable square-QDM product unit cell was found.")

repeatable_report_index, product_unit_cell, product_sequence = repeatable_candidates[0]
{
    "is_certified": product_sequence.is_certified,
    "repeat_axis": product_unit_cell.repeat_axis,
    "energy_density": product_sequence.energy_density,
    "support_size_per_unit_cell": product_unit_cell.support_size_per_unit_cell,
    "unit_cell_winding_sector": product_sequence.unit_cell_winding_sector,
    "verification_repeats": product_sequence.verification_repeats,
}

In [ ]:
product_scaling = scan_square_qdm_periodic_product_cancellation_scaling(
    product_unit_cell,
    repeat_counts=(1, 2, 3),
    max_support_size=128,
    tolerance=1.0e-9,
)
product_scaling_table = pd.DataFrame([
    point.to_summary_dict() for point in product_scaling.points
])
product_scaling_table.to_csv(DATA_DIR / "qdm_4N_by_4_exact_sequence.csv", index=False)
display(product_scaling_table[[
    "repeats", "system_size", "support_size", "shell_size",
    "boundary_nullity", "interference_gap", "product_state_boundary_residual",
    "kinetic_constraint_rank", "kinetic_compatible_dimension",
    "kinetic_compatible_fraction", "potential_constraint_rank",
]])

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(product_scaling_table["repeats"], product_scaling_table["interference_gap"], marker="o")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel(r"Boundary singular gap $\Delta_B$")
ax.set_xticks(product_scaling_table["repeats"])
save_figure(fig, "qdm_strip_interference_gap")
plt.show()

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(product_scaling_table["repeats"], product_scaling_table["kinetic_constraint_rank"], marker="o", label="compatibility rank")
ax.plot(product_scaling_table["repeats"], [16*n for n in product_scaling_table["repeats"]], marker="o", label="local kinetic parameters")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel("Dimension")
ax.set_xticks(product_scaling_table["repeats"])
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_compatibility_scaling")
plt.show()

The exact product sequence has support $4^N$, boundary nullity one, and interference gap two on the explicit sizes.  Its preserving kinetic constraints have rank $2N$ out of $16N$ plaquette couplings.  Therefore the exact sequence is size extensible but its generic local-deformation compatibility has an **extensive codimension**.  This is not finite-codimension topological protection.

## 6. Fixed local witness and beta-zero strip activity

A reduced-IZ witness is extracted from the same certified compact record, normalized to $\|Q_R\|=1$, and propagated through the infinite product-sequence certificate.  The transfer calculation evaluates the exact beta-zero expectation in the $(W_x,W_y)=(0,0)$ sector without diagonalizing the exponentially growing strip Hilbert space.

In [ ]:
stripe_record = stripe_certified.records[repeatable_report_index]
stripe_classification = classify_cage_state(
    stripe_record.cage_state,
    kinetic_matrix=stripe_certified.kinetic_matrix,
    basis_configs=stripe_certified.basis.states,
    hilbert_size=stripe_certified.hilbert_size,
    config=CageClassificationConfig(sector_policy="infer_support_component"),
)
stripe_witnesses = local_witnesses_from_classification_report(stripe_classification)
sequence_witness = certify_local_witness_on_square_qdm_periodic_sequence(
    product_sequence,
    stripe_witnesses[0],
    normalization="operator_norm",
)

strip_lengths = (4, 8, 12, 16, 24, 32, 48, 64, 96, 128)
strip_witness_report = evaluate_square_qdm_classification_witnesses_on_strips(
    stripe_classification,
    model=square_model,
    lengths=strip_lengths,
    winding_sector=(0, 0),
    normalization="operator_norm",
    winding_projection="fourier",
)
selected_strip_witness = strip_witness_report.records[0]

strip_witness_table = pd.DataFrame([
    evaluation.to_summary_dict()
    for evaluation in selected_strip_witness.scaling_report.evaluations
])
strip_witness_table["cage_expectation"] = 0.0
strip_witness_table.to_csv(DATA_DIR / "qdm_strip_witness_beta_zero.csv", index=False)

{
    "sequence_witness": sequence_witness.to_summary_dict(),
    "n_available_witnesses": len(strip_witness_report.records),
    "selected_window_width": selected_strip_witness.placement.window_width,
    "selected_link_coordinates": selected_strip_witness.placement.link_coordinates,
    "tail_estimate": selected_strip_witness.scaling_report.tail_estimate(),
}

In [ ]:
display(strip_witness_table[[
    "length", "circumference", "expectation", "cage_expectation",
    "partition_count", "window_width",
]])

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(strip_witness_table["length"], strip_witness_table["expectation"], marker="o", label=r"$\beta=0$, $(0,0)$ sector")
ax.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel(r"$\mathrm{Tr}(\rho_{\beta=0}Q_R)$")
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_witness_activity")
plt.show()

The thermal activity remains positive and approaches about $0.09$ for this normalized witness.  This supplies a rigorous beta-zero lower bound for the **width-four strip ensemble**.  It does not by itself place the uniform RK cage at beta zero.

## 7. Energy-density matching and the pure-kinetic trap

For the uniform RK potential, the repeated cage has $e_{\rm cage}=1/4$, while the width-four beta-zero energy density approaches a slightly larger value.  Thus the uniform model requires a finite-temperature comparison.  Setting the potential coupling to zero gives exact beta-zero matching, but produces a large chiral zero-mode manifold on the finite torus and contaminates level statistics near the cage energy.

In [ ]:
energy_lengths = (4, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256)
beta_zero_energy_report = scan_square_qdm_beta_zero_energy_density(
    tuple((length, 4) for length in energy_lengths),
    potential_coupling=1.0,
    winding_sector=(0, 0),
    winding_projection="fourier",
)
energy_table = pd.DataFrame([
    {
        "length": evaluation.length,
        "circumference": evaluation.circumference,
        "beta_zero_energy_density": evaluation.energy_density,
        "uniform_cage_energy_density": product_sequence.energy_density,
        "mismatch": evaluation.energy_density - product_sequence.energy_density,
    }
    for evaluation in beta_zero_energy_report.evaluations
])
energy_table.to_csv(DATA_DIR / "qdm_strip_beta_zero_energy_density.csv", index=False)
display(energy_table)

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(energy_table["length"], energy_table["beta_zero_energy_density"], marker="o", label=r"$e_{\beta=0}$")
ax.axhline(product_sequence.energy_density, linestyle="--", linewidth=0.8, label=r"uniform cage $e=1/4$")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel("Energy density")
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_energy_matching")
plt.show()

In [ ]:
kinetic_eigenvalues = scipy_linalg.eigvalsh(square_build.kinetic.toarray())
kinetic_zero_count = int(np.sum(np.abs(kinetic_eigenvalues) <= RANK_TOL))
kinetic_zero_fraction = kinetic_zero_count / kinetic_eigenvalues.size
kinetic_gap_report = adjacent_gap_ratio_report(
    kinetic_eigenvalues,
    trim_fraction=0.1,
    degeneracy_tolerance=RANK_TOL,
)
{
    "pure_kinetic_beta_zero_match": True,
    "zero_mode_count_4x4_w00": kinetic_zero_count,
    "zero_mode_fraction": kinetic_zero_fraction,
    "usable_gap_ratios_after_degeneracy_filter": len(kinetic_gap_report.ratios),
    "mean_adjacent_gap_ratio": kinetic_gap_report.mean_ratio,
}

The pure-kinetic point is therefore useful for an exact energy-density identity but poor as the sole thermal-background demonstration.  A finite or inhomogeneous diagonal term is needed to lift accidental zero modes while preserving the cage.

## 8. Finite-size beta-zero-matched inhomogeneous control

Every plaquette flippability projector has a definite eigenvalue on the compact cage.  Therefore arbitrary plaquette-dependent potential coefficients preserve the exact state.  On the $4\times4$ torus, a single linear constraint matches its energy to the beta-zero trace.  We project a documented random coefficient vector onto this 15-dimensional matching space.  The inhomogeneity breaks translations and point-group symmetries and removes the pure-kinetic zero manifold.

This is a finite-size thermal-background control, not yet a full strip deformation theorem.

In [ ]:
if len(potential_term_matrices) != 2 * len(square_model.plaquette_ids()):
    raise RuntimeError("Expected two orientation projectors per square plaquette.")
plaquette_potential_matrices = tuple(
    potential_term_matrices[2 * index] + potential_term_matrices[2 * index + 1]
    for index in range(len(square_model.plaquette_ids()))
)

compact_state = states_04[:, 0]
scar_flippabilities = np.asarray([
    np.vdot(compact_state, matrix @ compact_state).real
    for matrix in plaquette_potential_matrices
])
finite_beta_zero_flippabilities = np.asarray([
    np.mean(matrix.diagonal().real)
    for matrix in plaquette_potential_matrices
])
finite_match = beta_zero_matching_subspace(
    scar_flippabilities,
    finite_beta_zero_flippabilities,
    tolerance=TOL,
)

rng = np.random.default_rng(RANDOM_SEED)
matched_coefficients = project_coefficients_to_beta_zero_match(
    rng.normal(size=len(plaquette_potential_matrices)),
    finite_match,
    normalize=True,
) * 4.0

matched_hamiltonian = square_build.kinetic.astype(np.complex128)
for coefficient, matrix in zip(matched_coefficients, plaquette_potential_matrices, strict=True):
    matched_hamiltonian = matched_hamiltonian + coefficient * matrix

matched_eigenvalues, matched_eigenvectors = scipy_linalg.eigh(matched_hamiltonian.toarray())
matched_scar_report = diagnose_eigenpair(matched_hamiltonian, compact_state)
matched_gap_report = adjacent_gap_ratio_report(
    matched_eigenvalues,
    trim_fraction=0.1,
    degeneracy_tolerance=RANK_TOL,
)

full_classification = classify_cage_state(
    records_04[0].cage_state,
    kinetic_matrix=square_build.kinetic,
    basis_configs=square_build.basis.states,
    hilbert_size=square_search.hilbert_size,
    config=CageClassificationConfig(sector_policy="infer_support_component"),
)
full_witness = local_witnesses_from_classification_report(full_classification)[0]
normalized_template = full_witness.template.normalized("operator_norm")
normalized_witness = normalized_template.instantiate(full_witness.variable_indices)
local_row_operator = normalized_witness.embed(square_build.basis.states)
q_operator = (local_row_operator.conj().T @ local_row_operator).toarray()

scar_overlaps = np.abs(matched_eigenvectors.conj().T @ compact_state) ** 2
scar_index = int(np.argmax(scar_overlaps))
q_expectations = eigenstate_expectations(q_operator, matched_eigenvectors)
window = select_microcanonical_window_by_count(
    matched_eigenvalues,
    target_energy=float(matched_scar_report.energy.real),
    target_count=24,
    exclude_indices=(scar_index,),
    include_boundary_degeneracy=True,
    degeneracy_tolerance=TOL,
)
window_indices = np.asarray(window.indices, dtype=np.int64)

matched_summary = {
    **finite_match.to_summary_dict(),
    "scar_energy": float(matched_scar_report.energy.real),
    "beta_zero_trace_energy": float(np.trace(matched_hamiltonian.toarray()).real / square_search.hilbert_size),
    "scar_residual": matched_scar_report.residual_norm,
    "scar_overlap": float(scar_overlaps[scar_index]),
    "zero_mode_count": int(np.sum(np.abs(matched_eigenvalues) <= RANK_TOL)),
    "mean_gap_ratio": matched_gap_report.mean_ratio,
    "microcanonical_state_count": window.n_states,
    "microcanonical_half_width": window.half_width,
    "microcanonical_center_offset": window.center_offset,
    "microcanonical_q_mean": float(np.mean(q_expectations[window_indices])),
    "microcanonical_q_minimum": float(np.min(q_expectations[window_indices])),
    "scar_q_expectation": float(q_expectations[scar_index]),
}
pd.DataFrame([matched_summary]).to_csv(DATA_DIR / "qdm_4x4_beta_zero_matched_control.csv", index=False)
pd.DataFrame({
    "plaquette": np.arange(len(matched_coefficients)),
    "coefficient": matched_coefficients,
    "scar_flippability": scar_flippabilities,
    "finite_beta_zero_flippability": finite_beta_zero_flippabilities,
}).to_csv(DATA_DIR / "qdm_4x4_beta_zero_matched_coefficients.csv", index=False)
matched_summary

In [ ]:
spectral_table = pd.DataFrame({
    "energy": matched_eigenvalues,
    "energy_density": matched_eigenvalues / 16.0,
    "q_expectation": q_expectations,
    "is_scar": np.arange(matched_eigenvalues.size) == scar_index,
    "is_microcanonical": np.isin(np.arange(matched_eigenvalues.size), window_indices),
})
spectral_table.to_csv(DATA_DIR / "qdm_4x4_matched_eth_scatter.csv", index=False)

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.scatter(spectral_table["energy_density"], spectral_table["q_expectation"], s=10, alpha=0.65, label="sector eigenstates")
ax.scatter(
    [spectral_table.loc[scar_index, "energy_density"]],
    [spectral_table.loc[scar_index, "q_expectation"]],
    marker="*", s=75, label="compact cage",
)
ax.set_xlabel("Energy density")
ax.set_ylabel(r"$\langle Q_R\rangle$")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_matched_eth_scatter")
plt.show()

For this documented finite-size control, the exact cage sits at the beta-zero trace energy, has zero local-witness activity, and is surrounded by states with positive $\langle Q_R\rangle$.  The mean adjacent-gap ratio is closer to GOE than to Poisson, but the Hilbert space contains only 132 states.  The result should be reported as corroborating finite-size evidence, not as a thermodynamic chaos proof.

## 9. Quasi-one-dimensional audit

The following report makes the limitations machine-readable.  It combines exact sequence data, fixed-width witness activity, the linearly growing compatibility rank, the uniform-model energy mismatch, and the pure-kinetic zero-mode problem.

In [ ]:
strip_evaluation_by_length = {
    evaluation.length: evaluation
    for evaluation in selected_strip_witness.scaling_report.evaluations
}
scaling_points = []
for point in product_scaling.points:
    length = int(point.system_size[0])
    thermal_evaluation = strip_evaluation_by_length[length]
    scaling_points.append(Quasi1DSequencePoint(
        length=length,
        width=int(point.system_size[1]),
        exact_residual=float(point.product_state_boundary_residual),
        witness_radius=float(selected_strip_witness.placement.window_width),
        transverse_witness_span=float(selected_strip_witness.placement.circumference),
        thermal_second_moment=float(thermal_evaluation.expectation),
        interference_gap=float(point.interference_gap),
        compatibility_rank=int(point.kinetic_compatibility.rank),
        local_parameter_count=16 * int(point.repeats),
        support_size=int(point.support_size),
        sector_dimension=int(round(thermal_evaluation.partition_count)),
    ))

uniform_energy_mismatch = float(
    beta_zero_energy_report.evaluations[-1].energy_density
    - product_sequence.energy_density
)
quasi_1d_audit = audit_quasi_1d_sequence(
    scaling_points,
    energy_density_mismatch=uniform_energy_mismatch,
    level_gap_ratio=matched_gap_report.mean_ratio,
    zero_mode_fraction=kinetic_zero_fraction,
    tolerance=1.0e-8,
)

pd.DataFrame([point.to_summary_dict() for point in scaling_points]).to_csv(
    DATA_DIR / "qdm_quasi_1d_audit_points.csv", index=False
)
pd.DataFrame({
    "kind": ["established"] * len(quasi_1d_audit.established)
        + ["problem"] * len(quasi_1d_audit.issues),
    "statement": list(quasi_1d_audit.established) + list(quasi_1d_audit.issues),
}).to_csv(DATA_DIR / "qdm_quasi_1d_audit_statements.csv", index=False)

print("Established:")
for statement in quasi_1d_audit.established:
    print("  +", statement)
print("\nProblems / limitations:")
for statement in quasi_1d_audit.issues:
    print("  -", statement)

### Interpretation of the quasi-1D result

The fixed-width calculation can support the following conservative statement:

> There exists an exact $(4N)\times4$ square-QDM cage sequence in the zero-winding sector, with a system-size-independent reduced-IZ witness whose beta-zero second moment remains positive in the width-four strip ensemble.

Several qualifications remain essential:

- **Not a 2D limit.** Width four defines an effective one-dimensional constrained system. Its transfer spectrum, phases, and ETH properties need not approach those of the 2D QDM.
- **Transverse locality is not tested.** A witness is bounded along the sequence, but because the circumference never increases it can occupy a finite fraction of the transverse ring. The calculation does not prove that the same construction remains local as $L_y$ grows.
- **The uniform cage is not at beta zero.** For $\lambda=1$, the strip beta-zero energy density tends to a value above $1/4$, so a finite-temperature thermal calculation is still required for the undeformed RK Hamiltonian.
- **The obvious beta-zero fix is spectrally singular.** At $\lambda=0$, exact energy matching coexists with a large chiral zero-mode manifold; level statistics and microcanonical windows at the scar energy become unreliable.
- **Protection has extensive codimension.** The repeated compact cage requires two kinetic equality constraints per unit cell. The exact family is size extensible but not stable to a finite-codimension class of generic local kinetic perturbations.
- **The ninth mode has no sequence here.** On $4\times4$ it requires all 16 plaquette actions in the tested basis. No bounded-radius or repeated nonfactorized continuation is established in this notebook.
- **Thin-torus sector effects.** The $(0,0)$ winding sector and finite circumference can produce parity oscillations and atypical local probabilities. Transfer-matrix tail spreads should be shown rather than hidden by a single fit.

These limitations do not invalidate the quasi-1D ETH witness. They delimit its scope and prevent it from being presented as a true two-dimensional or topological thermodynamic result.

## 10. Optional expensive collective-extension scan

The codebase contains a finite-range column-grammar search that asks whether the collective $4\times4$ record extends to a nonfactorized width-four cage outside the translated product span.  It can be much more expensive than the main notebook and is therefore disabled by default.  A positive `locality_extension_index` would be genuinely new; a zero result is only a no-go within the tested grammar.

In [ ]:
RUN_EXPENSIVE_COLLECTIVE_EXTENSION = False

if RUN_EXPENSIVE_COLLECTIVE_EXTENSION:
    collective_record = records_04[8]
    collective_support_configs = np.asarray(
        [square_build.basis.state(int(index)) for index in collective_record.cage_state.support],
        dtype=np.int64,
    )
    collective_extension = scan_square_qdm_collective_locality_extension(
        square_model,
        collective_support_configs,
        product_unit_cell,
        cases=((2, 8), (3, 8), (3, 12)),
        potential_per_column=1.0,
        max_words=10_000,
        max_product_support_size=128,
        dense_column_limit=512,
        maximum_nullity=16,
        ipr_restarts=32,
        tolerance=1.0e-9,
    )
    collective_extension_table = pd.DataFrame([
        point.to_summary_dict() for point in collective_extension.points
    ])
    collective_extension_table.to_csv(
        DATA_DIR / "qdm_collective_locality_extension.csv", index=False
    )
    display(collective_extension_table)
else:
    print("Skipped. Set RUN_EXPENSIVE_COLLECTIVE_EXTENSION=True for the finite-range grammar scan.")

## Export manifest

In [ ]:
manifest = sorted(
    str(path.relative_to(REPO_ROOT))
    for path in DATA_DIR.rglob("*")
    if path.is_file()
)
for item in manifest:
    print(item)